# Increment 1 — Data pipeline and YOLO tumour detector
Cleans and partitions the data (Section 3.5) and trains the YOLO detector (Step 1). Produces Table 4.5. Then **Save Version** so the output can be attached to notebook 02.

In [ ]:
# === STEP 0: SETUP — run this cell first, every time =========================
# Before running: in the right-hand panel
#   * "Session options" -> Accelerator = GPU T4 x2, Internet = On
#   * "Add Input" -> your code dataset (the uploaded bcd-project.zip)
#   * "Add Input" -> search "brain tumor mri dataset" by masoudnickparvar
#   * (notebooks 02-05) "Add Input" -> "Your Work" -> the previous notebook(s)
import os, sys, glob, shutil, subprocess

def find_dir(test, root='/kaggle/input', max_depth=6):
    # walk the attached inputs and return the first folder that passes test(folder)
    root_depth = root.rstrip('/').count('/')
    for d, subdirs, files in os.walk(root):
        if d.count('/') - root_depth > max_depth:
            subdirs[:] = []
            continue
        if test(d, subdirs, files):
            return d
    return None

# 1) the project code: the folder that contains bcd/__init__.py
code_dir = find_dir(lambda d, s, f: os.path.isfile(os.path.join(d, 'bcd', '__init__.py')))
if code_dir is None:
    raise SystemExit('Code not found. Click "Add Input" and attach the dataset you created from bcd-project.zip.')
shutil.copytree(code_dir, '/kaggle/working/bcd-project', dirs_exist_ok=True)
sys.path.insert(0, '/kaggle/working/bcd-project')
print('code found at:', code_dir)

# 2) install the extra libraries (takes ~1-2 minutes)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45,<4.52', 'peft>=0.12',
                'bitsandbytes>=0.43', 'accelerate>=0.33', 'ultralytics', 'imagehash', 'h5py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])  # old torchao on Kaggle breaks peft

# 3) the brain MRI dataset: the folder that has Training/ and Testing/ inside
from bcd.config import load_config
cfg = load_config('/kaggle/working/bcd-project/configs/default.yaml')
mri = find_dir(lambda d, s, f: 'Training' in s and 'Testing' in s)
if mri is None:
    raise SystemExit('MRI dataset not found. Click "Add Input" and attach masoudnickparvar/brain-tumor-mri-dataset.')
cfg.paths.kaggle_root = mri
print('MRI dataset found at:', mri)

# 4) results of earlier notebooks (a folder named bcd that contains manifest.csv)
os.makedirs(cfg.paths.work_dir, exist_ok=True)
for prev in glob.glob('/kaggle/input/**/bcd/manifest.csv', recursive=True):
    src = os.path.dirname(prev)
    if not src.startswith('/kaggle/working'):
        shutil.copytree(src, cfg.paths.work_dir, dirs_exist_ok=True)
        print('restored earlier results from:', src)
print('GPUs available:', __import__('torch').cuda.device_count())
print('Setup complete. Work folder:', cfg.paths.work_dir)


In [ ]:
# Figshare .mat files (Cheng et al., 2015) are needed for YOLO and the A_XAI reference masks
from bcd.data import find_figshare_dir
cfg.paths.figshare_root = find_figshare_dir('/kaggle/input')
n_mat = len(glob.glob(f'{cfg.paths.figshare_root}/**/*.mat', recursive=True)) if cfg.paths.figshare_root else 0
print('figshare folder:', cfg.paths.figshare_root, '| .mat files:', n_mat)
if n_mat < 3000:
    raise SystemExit('Figshare DATASET not attached. Add Input -> Datasets tab -> search "figshare brain tumor dataset" -> add the dataset with 3064 .mat files (a dataset, not a notebook).')
# To save GPU time you can cap the stream, e.g. cfg.data.stream_limit = 1500

In [ ]:
from bcd.data import prepare_all
man = prepare_all(cfg)
import pandas as pd
print(pd.read_csv(f'{cfg.paths.work_dir}/partition_summary.csv', index_col=0))
print('duplicates removed:', len(pd.read_csv(f'{cfg.paths.work_dir}/duplicates_removed.csv')))
print('images with reference masks:', man.mask_path.notna().sum())

In [ ]:
from bcd.detector import TumourDetector
weights = TumourDetector.train(f'{cfg.paths.work_dir}/yolo/data.yaml', cfg.yolo.weights, cfg.yolo.epochs,
                               cfg.yolo.imgsz, cfg.yolo.batch, project=f'{cfg.paths.work_dir}/yolo_runs', device=0)
det = TumourDetector(weights, cfg.yolo.conf_keep, cfg.yolo.imgsz, device=0)
t45 = det.validate(f'{cfg.paths.work_dir}/yolo/data.yaml')
import json; json.dump(t45, open(f'{cfg.paths.work_dir}/table_4_5_yolo.json','w'), indent=2)
t45  # -> Table 4.5 ('All' row; single tumour class)